# معمل تجارب توسيع الاستعلامات (Query Expansion Lab)

في هذا اللاب التفاعلي، سنتعرف عمليا على تقنية **توسيع الاستعلامات (Query Expansion)**، وهي أولى تقنيات تحسين الاستعلامات في مسار الـ RAG.

فكرة التقنية ببساطة:
المستخدم غالبا يكتب استعلامات مقتضبة جدا أو غامضة (مثل `langchain memory` أو `tools in llm`). لو أرسلنا هذا الاستعلام مباشرة للمسترجع، قد تضيع منا المقاطع المهمة.
الحل أن نمرر الاستعلام على نموذج لغوي مع قالب توجيه ذكي يعيد صياغة السؤال ويثريـه بالمترادفات والمفاهيم التقنية قبل البحث في قاعدة المتجهات.

### 1. تجهيز قاعدة البيانات المصغرة
تعال نجهز قاعدة بيانات بسيطة تحتوي على شروحات تقنية عن مكتبة LangChain:

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# تحميل نموذج التضمين المحلي المجاني
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# قاعدة وثائق تجريبية
docs = [
    Document(
        page_content="توفر مكتبة LangChain وحدات إدارة الذاكرة مثل ConversationBufferMemory لحفظ سجل المحادثة كاملا بين المستخدم والنموذج.",
        metadata={"topic": "memory_buffer"}
    ),
    Document(
        page_content="تستخدم ConversationSummaryBufferMemory لتلخيص الحوارات القديمة بواسطة LLM مع الاحتفاظ بآخر الرسائل لتوفير التوكنز في الجلسات الطويلة.",
        metadata={"topic": "memory_summary"}
    ),
    Document(
        page_content="تعتمد الذاكرة المتجهية VectorStoreRetrieverMemory على تخزين المحادثات السابقة في قاعدة بيانات واسترجاع الأجزاء ذات الصلة دلاليا فقط.",
        metadata={"topic": "memory_vector"}
    ),
    Document(
        page_content="تتيح واجهات StructuredTool و ToolNode للوكلاء استدعاء أدوات خارجية مثل الآلات الحاسبة وقواعد البيانات وواجهات برمجة التطبيقات APIs.",
        metadata={"topic": "tool_calling"}
    ),
]

vector_store = FAISS.from_documents(docs, embeddings)
print("تم بناء الفهرس بنجاح بإجمالي مستندات:", vector_store.index.ntotal)

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_27596\1667673958.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء الفهرس بنجاح بإجمالي مستندات: 4


### 2. تجربة البحث العادي بالاستعلام المقتضب (Naive Query)
لنفترض أن المستخدم كتب كلمتين فقط: `langchain memory`
تعال نشوف المستندات المسترجعة والمسافات الدلالية:

In [2]:
raw_query = "langchain memory"

naive_results = vector_store.similarity_search_with_score(raw_query, k=2)

print(f"الاستعلام الخام: '{raw_query}'\n")
print("النتائج المسترجعة بالبحث العادي:")
for rank, (doc, score) in enumerate(naive_results, 1):
    print(f"[{rank}] المحور: {doc.metadata['topic']} (المسافة: {score:.4f})")
    print(f"المحتوى: {doc.page_content}\n")

الاستعلام الخام: 'langchain memory'

النتائج المسترجعة بالبحث العادي:
[1] المحور: memory_buffer (المسافة: 1.3651)
المحتوى: توفر مكتبة LangChain وحدات إدارة الذاكرة مثل ConversationBufferMemory لحفظ سجل المحادثة كاملا بين المستخدم والنموذج.

[2] المحور: memory_vector (المسافة: 1.5827)
المحتوى: تعتمد الذاكرة المتجهية VectorStoreRetrieverMemory على تخزين المحادثات السابقة في قاعدة بيانات واسترجاع الأجزاء ذات الصلة دلاليا فقط.



### 3. بناء سلسلة توسيع الاستعلام (Query Expansion Chain)
دلوقتي هنبني سلسلة بسيطة تأخذ السؤال القصير، وتطلب من النموذج اللغوي توسيعه بالمصطلحات التقنية والمترادفات.
لاحظ القالب: بنمنع النموذج من الإجابة عن السؤال، وبنطلب منه فقط إضافة الكلمات المفتاحية ذات الصلة:

In [3]:
expansion_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "أنت خبير في تحسين استعلامات البحث لأنظمة RAG التقنية.\n"
        "مهمتك: إعادة صياغة وتوسيع استعلام المستخدم بإضافة المترادفات والمفاهيم التقنية المرتبطة به.\n"
        "تعليمات صارمة:\n"
        "1. لا تجب على السؤال إطلاقا.\n"
        "2. اكتب استعلاما موسعا وغنيا بالكلمات المفتاحية فقط."
    ),
    ("human", "قم بتوسيع استعلام البحث التالي: {query}")
])

# تجهيز محرك التحويل
groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
        expansion_chain = expansion_prompt | llm | StrOutputParser()
        expanded_query = expansion_chain.invoke({"query": raw_query})
    except Exception:
        # محاكاة في حالة عدم توفر الاتصال
        expanded_query = "LangChain conversation memory modules ConversationBufferMemory summary buffer window chat history"
else:
    # محاكاة تلقائية
    expanded_query = "LangChain conversation memory modules ConversationBufferMemory summary buffer window chat history"

print("الاستعلام الأصلي:", raw_query)
print("الاستعلام الموسع:", expanded_query)

الاستعلام الأصلي: langchain memory
الاستعلام الموسع: langchain memory OR “ذاكرة LangChain” OR “تخزين LangChain” OR “مستودع LangChain” OR “مخزن LangChain” OR “ذاكرة مؤقتة LangChain” OR “ذاكرة دائمة LangChain” OR “مستودع الذاكرة LangChain” OR “مستودع الذاكرة المؤقتة LangChain” OR “مستودع الذاكرة المتزامنة LangChain” OR “مستودع الذاكرة المتعددة اللغات LangChain” OR “مستودع الذاكرة المتكاملة LangChain” OR “مستودع الذاكرة المتقدمة LangChain” OR “مستودع الذاكرة المتزامنة LangChain” OR “مستودع الذاكرة المتعددة اللغات LangChain” OR “مستودع الذاكرة المتكاملة LangChain” OR “مستودع الذاكرة المتقدمة LangChain” OR “مستودع الذاكرة المتزامنة LangChain” OR “مستودع الذاكرة المتعددة اللغات LangChain” OR “مستودع الذاكرة المتكاملة LangChain


### 4. البحث باستخدام الاستعلام الموسع (Enhanced Retrieval)
تعال نشوف المستندات المسترجعة لما نبحث بالاستعلام الغني بعد التوسيع:

In [4]:
enhanced_results = vector_store.similarity_search_with_score(expanded_query, k=2)

print("نتائج البحث بالاستعلام الموسع:")
for rank, (doc, score) in enumerate(enhanced_results, 1):
    print(f"[{rank}] المحور: {doc.metadata['topic']} (المسافة: {score:.4f})")
    print(f"المحتوى: {doc.page_content}\n")

نتائج البحث بالاستعلام الموسع:
[1] المحور: memory_buffer (المسافة: 0.9929)
المحتوى: توفر مكتبة LangChain وحدات إدارة الذاكرة مثل ConversationBufferMemory لحفظ سجل المحادثة كاملا بين المستخدم والنموذج.

[2] المحور: memory_summary (المسافة: 1.1996)
المحتوى: تستخدم ConversationSummaryBufferMemory لتلخيص الحوارات القديمة بواسطة LLM مع الاحتفاظ بآخر الرسائل لتوفير التوكنز في الجلسات الطويلة.



### 5. مقارنة شمولية الاستعلامين
تعال نقارن عدد الكلمات المشتركة والتشابه بين الاستعلام الأصلي والاستعلام الموسع مقارنة بالوثائق المستهدفة:

In [5]:
def check_overlap(query, target_text):
    q_words = set(query.lower().split())
    t_words = set(target_text.lower().split())
    common = q_words.intersection(t_words)
    return len(common), common

target = docs[0].page_content

raw_count, raw_common = check_overlap(raw_query, target)
exp_count, exp_common = check_overlap(expanded_query, target)

print("مقارنة الكلمات المشتركة مع الوثيقة المستهدفة:")
print(f"- الاستعلام الأصلي: {raw_count} كلمات مشتركة -> {raw_common}")
print(f"- الاستعلام الموسع: {exp_count} كلمات مشتركة -> {exp_common}")
print(f"النتيجة: زيادة في تغطية الكلمات المفتاحية بمقدار +{exp_count - raw_count} كلمات!")

مقارنة الكلمات المشتركة مع الوثيقة المستهدفة:
- الاستعلام الأصلي: 1 كلمات مشتركة -> {'langchain'}
- الاستعلام الموسع: 2 كلمات مشتركة -> {'langchain', 'الذاكرة'}
النتيجة: زيادة في تغطية الكلمات المفتاحية بمقدار +1 كلمات!


### 6. جرب بنفسك!
الآن حان دورك للتجربة الحرة:
1. جرب استعلاما مقتضبا آخر مثل: `tools in llm`
2. لاحظ كيف سيقوم التوسيع بإضافة مصطلحات مثل `StructuredTool` و `ToolNode` و `APIs` لجلب المستند الرابع فورا!

In [6]:
# اكتب استعلامك التجريبي هنا
my_test_query = "tools in llm"

# توسيع الاستعلام يدويا أو عبر السلسلة
if groq_key:
    try:
        my_expanded = expansion_chain.invoke({"query": my_test_query})
    except Exception:
        my_expanded = "LangChain agents StructuredTool ToolNode API calculators tool calling"
else:
    my_expanded = "LangChain agents StructuredTool ToolNode API calculators tool calling"

print("الاستعلام التجريبي:", my_test_query)
print("الاستعلام الموسع:", my_expanded)

# البحث بالاستعلام الجديد
test_results = vector_store.similarity_search(my_expanded, k=1)
print("\nأهم مقطع تم استرجاعه:")
print(f"[{test_results[0].metadata['topic']}]: {test_results[0].page_content}")

الاستعلام التجريبي: tools in llm
الاستعلام الموسع: LLM tools, large language model tools, LLM toolkits, LLM utilities, LLM plugins, LLM APIs, LLM frameworks, LLM integration, LLM extensions, LLM libraries, LLM toolchains, LLM tool development, LLM tool usage, LLM tool selection, LLM tool evaluation, LLM tool architecture, LLM tool automation, LLM tool deployment, LLM tool management, LLM tool ecosystem, LLM tool performance, LLM tool security, LLM tool compliance, LLM tool best practices, LLM tool documentation, LLM tool tutorials, LLM tool examples, LLM tool comparison, LLM tool features, LLM tool capabilities, LLM tool use cases, LLM tool applications, LLM tool NLP integration, LLM tool ML integration, LLM tool AI integration, LLM tool data pipeline integration, LLM tool cloud services integration, LLM tool open-source LLM integration, LLM tool commercial LLM integration, LLM tool OpenAI integration, LLM tool HuggingFace integration, LLM tool Transformers integration, LLM tool ChatGP